# Deriving parameters based on radial profiles

In [ ]:
import numpy as np
from astropy.io import fits
import matplotlib.pyplot as plt
from dwarfs.photometry import photometry as phot
from dwarfs.photometry import utils

### Building an isophotal profile

Deriving parameters like concentration and half-life radius requires a radial curve of growth, which requires fitting an isophotal profile of the galaxy in question.  Below is a demonstration of how to do this using the software package included in this repository.

The example image used has been hand-masked of interloping sources, as well as one pointlike source chosen at random within the galaxy (for demonstration purposes).

In [ ]:
# Read in the masked image
imarr = fits.getdata('masked.fits')

# Convert it to a masked array
maskarr = np.ma.masked_array(imarr, mask=(imarr == -999))

In [ ]:
# Make a profile instance using the masked image
prof = phot.Profile(maskarr)

# Estimate the galaxy center
halfBoxWid = 20  # Determines the size of the box used in the center of the image for this estimate
x0, y0 = prof.initializeCenter(halfBoxWid)

# Show this on the image
lgIm = utils.ds9LogScale(maskarr)
plt.figure(figsize=(3, 3))
plt.imshow(lgIm, vmin=-3, vmax=1)
plt.plot(x0, y0, 'rx')

In [ ]:
# Now fit isophotes, first fixing no parameters
ellipse = prof.initializeEllipse(x0=x0, y0=y0, sma=20, pa=0, eps=0.5)
isoListVar = prof.fitEllipses(ellipse)  # Using default parameters for fitter

In [ ]:
# Showing these on the image
plt.figure(figsize=(4, 4))
prof.visualizeEllipses(isoListVar, vmin=-3, vmax=1)

In [ ]:
# Plotting some profiles
fig, ax = plt.subplots(2, 2, figsize=(8, 8))

ax[0, 0].plot(isoListVar.sma, np.degrees(isoListVar.pa), 'k.')
ax[0, 0].set_xlabel('Radius (px)')
ax[0, 0].set_ylabel('Position angle (degrees)')

ax[0, 1].plot(isoListVar.sma, isoListVar.eps, 'k.')
ax[0, 1].set_xlabel('Radius (px)')
ax[0, 1].set_ylabel('Ellipticity (1-b/a)')

ax[1, 0].plot(isoListVar.sma, isoListVar.x0, 'k.')
ax[1, 0].plot(0, x0, 'rx')  # The initial guess
ax[1, 0].set_xlabel('Radius (px)')
ax[1, 0].set_ylabel('Center x (px)')

ax[1, 1].plot(isoListVar.sma, isoListVar.y0, 'k.')
ax[1, 1].plot(0, y0, 'rx')  # The initial guess
ax[1, 1].set_xlabel('Radius (px)')
ax[1, 1].set_ylabel('Center y (px)')
plt.subplots_adjust(wspace=0.3, hspace=0.2)

Using fixed isophote shapes is better practice when deriving curves of growth to avoid e.g. ambiguity in semi-major axis length, &c.  The strategy employed for the various S4G iterations (e.g. Munoz-Mateos et al. 2015) is to fix isophote shapes based on galaxy outskirts, where the profiles often stabilize.  This works for early type galaxies and massive spirals, but irregular galaxies like our choice here evidently show more variability.  For demonstration purposes, we'll just use the mean of values derived beyond 50px in semi-major axis length, even if the "stability" seen in the last few points is a result of the fitter failing.

In [ ]:
# Define basic parameters
want = isoListVar.sma > 50
x0 = np.mean(isoListVar.x0[want])
y0 = np.mean(isoListVar.y0[want])
pa = np.mean(isoListVar.pa[want])
eps = np.mean(isoListVar.eps[want])
print(x0, y0, pa, eps)

Here we derive the noise in the image background as well, for later use interpolating flux across masks.  This is needed to preserve flux within masked pixels when measuring the curve of growth.

NOTE: this FFT-based azimuthal interpolation method is experimental and needs some refinement.

In [ ]:
# Getting estimates of the background noise and sky local to the galaxy
maxRad = isoListVar.sma[-1]*1.5  # For noise estimate; masks galaxy out to this radius
rms, sbLim, sky = phot.measureImageNoise(maskarr, x0, y0, maxRad, seed=12345)  # Using seed to keep results stable
print(rms, sbLim, sky)

In [ ]:
interparr = phot.interpolateAcrossMasks(maskarr, x0, y0, maxRad, pa, eps, rms)

lgIm = utils.ds9LogScale(interparr)
plt.figure(figsize=(3, 3))
plt.imshow(lgIm, vmin=-3, vmax=1)

Now we produce the fixed parameter profiles using this image.  This is done in PhotUtils by setting the maximum fitter radius parameter 'maxrit' to 1px.  We also set 'maxsma' (max radius out to which to fit) to speed up the process, and we use a set 0.5" width (3px) for all annuli rather than the default logarithmic behaviour.

In [ ]:
prof = phot.Profile(interparr)
ellipse = prof.initializeEllipse(x0=x0, y0=y0, sma=20, pa=pa, eps=eps)
isoList = prof.fitEllipses(ellipse, maxsma=isoListVar.sma[-1], maxrit=1, step=3, linear=True)

plt.figure(figsize=(4, 4))
prof.visualizeEllipses(isoList, vmin=-3, vmax=1)

In [ ]:
# Curve of growth and surface brightness profile
magZp = 27  # HSC, all photometric bands
pxScale = 0.168  # HSC pixel scale
rad = isoList.sma * pxScale
mag = -2.5*np.log10(isoList.tflux_e) + magZp
sb = -2.5*np.log10(isoList.intens) + magZp + 2.5*np.log10(pxScale**2)

fig, ax = plt.subplots(1, 2, figsize=(8, 3))

ax[0].plot(rad, mag, 'k.')
ax[0].set_xlabel('Radius (arcsec)')
ax[0].set_ylabel(r'$m(<R)$ (mag)')
ax[0].set_ylim([np.nanmax(mag)+0.5, np.nanmin(mag)-0.5])

ax[1].plot(rad, sb, 'k.')
ax[1].set_xlabel('Radius (arcsec)')
ax[1].set_ylabel(r'$\mu$ (mag arcsec$^{-2}$)')
ax[1].set_ylim([np.nanmax(sb)+0.5, np.nanmin(sb)-0.5])

The curve of growth is on the left, and the surface brightness profile on the right, using the outer isophote centre (which may not align with this galaxy's bulge).

### Deriving morphological parameters

In [ ]:
# Create an instance of a new class using the isoList in table format
# The `sky` parameter is our estimate of the local background correction, to remove systematics local to the galaxy
isoTab = prof.toTable(isoList)
dervals = phot.DerivedValues(isoTab, sky=sky, magZp=magZp, pxScale=pxScale)

In [ ]:
# Derive curve of growth corrected for local sky (see above)
sma, cog = dervals.curveOfGrowth()
# Use this to derive the total magnitude
totmag = dervals.totalMagnitude(sma, cog)
print("Total magnitude is: %.2f" % (totmag))

It's also always good practice to estimate uncertainty, which comes from photon counts (from source and sky), CCD read noise, and dark noise, e.g.: http://spiff.rit.edu/classes/ast613/lectures/signal/signal_illus.html

There is also uncertainty from the local sky background estimation, which we derive below

In [ ]:
# Derive uncertainty due to sky estimate
# N random perturbations to the local sky centered at the mean value, rederiving magnitude each time
N = 1000
rng = np.random.default_rng(12345)  # Using seed to keep results stable
randskies = rng.normal(sky, sbLim, size=N)  # Assuming sky obeys Gaussian statistics
randmags = []
for i in range(N):
    dervals1 = phot.DerivedValues(isoTab, sky=randskies[i], magZp=magZp, pxScale=pxScale)
    sma, cog = dervals1.curveOfGrowth()
    try:
        randmag = dervals1.totalMagnitude(sma, cog)
    except TypeError:
        randmags.append(np.nan)
        continue
    randmags.append(randmag)
magerr = np.nanstd(randmags)
print('Uncertainty on magnitude from sky subtraction: %.4f magnitudes' % (magerr))

#### Deriving integrated parameters

In [ ]:
# Using the fixed isophote profile
reff = dervals.fractionalRadius(totmag, fluxFrac=0.5)  # Half-light radius
rHolmberg = dervals.isophotalRadius(26.5)  # Holmberg radius
rHolmDeproj = dervals.isophotalRadiusDeproj(26.5, 1-eps)  # Deprojected for inclination
rpetro = dervals.petrosianRadius()
c82 = dervals.concentration82(totmag)  # C82 = 5log(R80/R20)
cRe = dervals.concentrationRe(totmag)  # From Trujillo et al. (2001)
muEffAv = dervals.muEffective(totmag)  # Av. surface brightness within half-light radius
muAtReff = dervals.muAtR(totmag, reff)  # Surface brightness at the half-light radius
n, muEff, rEff = dervals.iteratedSersicIndex(0, 30)
sRes = dervals.singleSersicResiduals(n, muEff, rEff, 0, 30)
print('Half-light radius = %.2f arcsec' % (reff*dervals.pxScale))
print('Isophotal radius for mu=26.5 mag/arcsec^2: %.2f arcsec' % (rHolmberg * dervals.pxScale))
print('As above, but for face-on projection: %.2f arcsec' % (rHolmDeproj * dervals.pxScale))
print('Petrosian radius = %.2f arcsec' % (rpetro*pxScale))
print('C82 = %.2f' % (c82))
print('C (Trujillo) = %.2f' % (cRe))
print('Average surface brightness within half-light radius: %.2f mag/arcsec^2' % (muEffAv))
print('Surface brightness at half-light radius: %.2f mag/arcsec^2' % (muAtReff))
print('Best-fit Sersic index, effective surface brightness, and half-light radius: %.2f, %.2f mag/arcsec^2, %.2f arcsec' % (n, muEff, rEff))
print('Mean sum of squares residuals from best-fit single Sersic profile: %.2f' % (sRes))

Note here that the concentration parameter here is approximately what it would be for an n=1 profile, and the Petrosian radius is around twice the half-light radius, also true for n=1 profiles.  The Sersic fit to the radial profile is <1 (typical of dwarfs), and the half-light radius corresponding to that fit isn't far from the empirical value derived using the curve of growth.

In [ ]:
# Using the variable isophote profiles
isoTabVar = prof.toTable(isoListVar)
dervalsVar = phot.DerivedValues(isoTabVar, sky=sky, magZp=magZp, pxScale=pxScale)

maxDelPa = dervalsVar.maxDeltaPa(sbLim)
twistiness = dervalsVar.twistiness(sbLim)
print('Maximum position angle swing (max - min PA): %.2f degrees' % (maxDelPa.value))
print('Twistiness value: %.2f' % (twistiness))

As expected, we see large swings in position angle across the profile, and the twistiness value is quite high (e.g. see Fig. 1 from Ryden et al. 1999).